# Demo: NBA Player Lookup, Keywords, And Video Formatting

This notebook resolves an NBA player name into a player ID, maps simple basketball words to NBA endpoint settings, fetches video details, and formats the raw playlist into a clean DataFrame.


In [ ]:
import importlib
import query

query = importlib.reload(query)

NBA_STATS_HEADERS = query.NBA_STATS_HEADERS
PLAYER_NAME = query.PLAYER_NAME
QUERY_TEXT = query.QUERY_TEXT
QUERY_PARAMS = query.QUERY_PARAMS
build_query_params = query.build_query_params
fetch_video_details = query.fetch_video_details
parse_keywords = query.parse_keywords
process_videos = query.process_videos
apply_keyword_filters = query.apply_keyword_filters
resolve_player = query.resolve_player
run_keyword_query = query.run_keyword_query


## Resolve a player

The NBA endpoint needs a numeric `player_id`. The notebook starts from a player name, then resolves it against the active-player list from `nba_api`.


In [ ]:
PLAYER_NAME


In [ ]:
resolve_player(PLAYER_NAME)


In [ ]:
resolve_player("cade")


## Parse basketball keywords

Simple keyword matching turns words like `dunks`, `misses`, `assists`, and `blocks` into NBA context measures and local filters.


In [ ]:
QUERY_TEXT


In [ ]:
keyword_params = parse_keywords(QUERY_TEXT)
keyword_params


In [ ]:
examples = ["dunks", "misses", "blocks", "assists", "threes", "stepback three"]
{example: parse_keywords(example) for example in examples}


## Build NBA query parameters

The base query keeps `team_id=0` so the player search is not tied to one current or historical team. `build_query_params()` copies the base parameters, injects the resolved `player_id`, and applies the parsed context measure.


In [ ]:
QUERY_PARAMS


In [ ]:
build_query_params(PLAYER_NAME, context_measure=keyword_params["context_measure"])


In [ ]:
NBA_STATS_HEADERS["User-Agent"]


## Fetch raw data

The raw response contains a playlist plus matching video URL metadata.


In [ ]:
video_details = fetch_video_details(PLAYER_NAME, context_measure=keyword_params["context_measure"])
video_details.keys()


In [ ]:
playlist = video_details["resultSets"]["playlist"]
video_urls = video_details["resultSets"]["Meta"]["videoUrls"]

len(playlist), len(video_urls)


## Format and filter results

`process_videos()` normalizes the raw response. `apply_keyword_filters()` handles local filters such as shot type and missed-shot detection.


In [ ]:
results = process_videos(video_details)
filtered_results = apply_keyword_filters(results, keyword_params)
results.shape, filtered_results.shape


In [ ]:
filtered_results[[
    "Game_ID",
    "Event_Index",
    "Game_Date",
    "Description",
    "Point_Change",
    "Score_Diff",
    "Score_Diff_After",
    "Video_Link",
    "Event_Link",
]].head(20)


## One-call helper


In [ ]:
quick_results = run_keyword_query(PLAYER_NAME, QUERY_TEXT)
quick_results.shape


## Save a sample


In [ ]:
output_path = "output/video_details_sample.csv"
filtered_results.to_csv(output_path, index=False)
output_path
